# Evidence-triggered retraining

This eighth notebook compares frozen models, fixed-schedule retraining and a one-sided CUSUM detector with delayed feedback, gradual concept drift and AR(1) dependence. The objective is not to maximise alarm frequency but to detect meaningful deterioration at acceptable maintenance cost.

## Sequential statistical evidence

The score is $S_t=-(Y_t-\\hat m_t)\\hat m_t/\\sigma$, using the frozen reference forecast. The one-sided statistic is $G_t=\\max(0,G_{t-1}+Z_t-\\kappa)$ where $Z_t$ is standardised using predeployment scores. A moving-block bootstrap estimates a heuristic threshold; it does not guarantee an anytime false-alarm rate. A label from step $j$ becomes usable at step $j+d+1$, never earlier.

In [ ]:
import os
import matplotlib.pyplot as plt
from IPython.display import display
from change_point_detection import run_detector_study, detector_summary
SMOKE = os.getenv("DETECTOR_SMOKE","0") == "1"
REPEATS = int(os.getenv("DETECTOR_REPEATS","6"))
if REPEATS < 1: raise ValueError("Invalid replicate count")
performance, decisions=run_detector_study(repeats=REPEATS,smoke=SMOKE)
summary=detector_summary(performance,decisions)
display(summary.round(3))
assert (decisions.regret >= -1e-9).all()

## False alarms, refits and decision regret

A stable stream provides a false-alarm control. A gradual-drift stream has no unique sharp change point. Compare alarm frequency, actual refits, prediction RMSE and economic regret across the same simulated streams and reporting delays. The economic oracle knows the synthetic conditional mean only at evaluation.

In [ ]:
focus=summary.loc[(summary.regime=="gradual_drift") & (summary.cost==0.6)]
fig,ax=plt.subplots(figsize=(8,4))
for policy,group in focus.groupby("policy"):
    group=group.sort_values("delay")
    ax.plot(group.delay,group.regret,marker="o",label=policy)
ax.set(xlabel="Outcome reporting delay",ylabel="Mean expected regret",title="Evidence-triggered versus scheduled retraining")
ax.legend()
fig.tight_layout()
plt.show()
display(focus[["delay","policy","rmse","alarms","refits","regret"]].round(3))

## Limitations

Historical block resampling does not establish sequential type-I error control. In particular, AR(1) dependence or noise shifts can generate spurious alarms, while changes orthogonal to frozen predictions may be missed. The default Monte Carlo run is exploratory and economic intervention effects are not identified causally. Next: compare Page–Hinkley/ADWIN, simulate more independent null streams and include explicit retraining costs.